# 05 · Decision Agent & the full Detect–Verify–Decide pipeline

**Purpose.** Fuse detector confidence with the verifier verdict via accept = (c ≥ τ_high) ∨ (v = yes ∧ c ≥ τ_low). Sweep (τ_low, τ_high) on held-out to trace the precision–recall frontier, select the operating point, and apply it unchanged to test. Optionally compares an LLM aggregator against the rule.

| | |
|---|---|
| **Proposal objective** | Objective 1 (Decision Agent), Objective 2 (pipeline vs single-pass detector) |
| **Inputs** | detection + verifier caches |
| **Outputs** | `results/tuning/selected.json` (τ_low, τ_high), `results/pipeline/*` |
| **Runtime (T4)** | seconds (offline from caches); optional LLM aggregator ≈10 min; demo ≈2 min |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, DRIVE_PROJECT], check=True)
    PROJECT_ROOT = DRIVE_PROJECT
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab already has torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from dvd.config import load_config, save_selected, results_dir
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache
from dvd import paths

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.data.coco import prepare_coco, CocoData, load_splits
prepare_coco(cfg)          # after a runtime restart this re-extracts from the zip cached on Drive (~1-2 min)
coco = CocoData(cfg)
splits = load_splits(cfg)
HELDOUT, TEST = splits["heldout"], splits["test"]
print(f"held-out: {len(HELDOUT)} images | test: {len(TEST)} images")

In [ ]:
from dvd.experiments import SplitData, available_methods, sweep, operating_point, evaluate_row, PROPOSED, DET_ONLY
from dvd.metrics import decision_breakdown
from dvd.viz import plot_pr
H = SplitData(cfg, coco, "heldout", HELDOUT); T = SplitData(cfg, coco, "test", TEST)
MH, MT = available_methods(cfg, H), available_methods(cfg, T)
assert PROPOSED in MH and PROPOSED in MT, "crop verifier caches missing - run notebook 03"

## 1. Threshold sweep on held-out

In [ ]:
base_rows = sweep(DET_ONLY, MH[DET_ONLY], H, cfg)
dvd_rows = sweep(PROPOSED, MH[PROPOSED], H, cfg)
sw = pd.DataFrame(dvd_rows)
heat = sw.pivot(index="tau_low", columns="tau_high", values="f1")
plt.figure(figsize=(9, 4)); plt.imshow(heat, cmap="viridis", aspect="auto")
plt.xticks(range(len(heat.columns)), heat.columns); plt.yticks(range(len(heat.index)), heat.index)
plt.xlabel("tau_high"); plt.ylabel("tau_low"); plt.colorbar(label="F1"); plt.title("Held-out F1 over the decision grid"); plt.show()

## 2. Select operating points (held-out only)
- **best-F1**: maximise F1.
- **matched-recall**: fewest false positives whose recall ≥ the detector-only best-F1 recall (the 'same recall, fewer FPs' view).

In [ ]:
base_best = operating_point(base_rows)
dvd_best = operating_point(dvd_rows)
dvd_match = operating_point(dvd_rows, "matched_recall", base_best["recall"])
save_selected(cfg, tau_low=dvd_best["tau_low"], tau_high=dvd_best["tau_high"],
              matched_tau_low=dvd_match["tau_low"], matched_tau_high=dvd_match["tau_high"],
              detector_only_tau=base_best["tau"])
cols = ["method", "tau", "tau_low", "tau_high", "precision", "recall", "f1", "fp_per_image", "n_verified"]
pd.DataFrame([base_best, dvd_best, dvd_match]).reindex(columns=cols)

## 3. Apply unchanged to test

In [ ]:
res = {"Detector only (tau*)": evaluate_row(MT[DET_ONLY], T, base_best, n_boot=1000),
       "DVD best-F1": evaluate_row(MT[PROPOSED], T, dvd_best, n_boot=1000),
       "DVD matched-recall": evaluate_row(MT[PROPOSED], T, dvd_match, n_boot=1000)}
fp0 = res["Detector only (tau*)"]["fp"]
tab = pd.DataFrame([{"config": k, **{c: v.get(c) for c in ["tau", "tau_low", "tau_high", "precision", "recall", "f1", "fp", "fp_per_image", "n_verified"]},
                     "FP reduction vs detector": 1 - v["fp"] / fp0, "F1 95% CI": v["ci"]["f1"]} for k, v in res.items()])
tab.to_csv(results_dir(cfg, "pipeline") / "test_operating_points.csv", index=False); tab

## 4. Decision-level breakdown on test
Reference pool = candidates the detector alone outputs at τ_low (what the pipeline starts from).

In [ ]:
br = []
for name in ["DVD best-F1", "DVD matched-recall"]:
    r = res[name]
    br.append({"config": name, **decision_breakdown(T.labels, {i: T.ev.scores(i) for i in T.ids}, r["_masks"], r["tau_low"])})
br_df = pd.DataFrame(br); br_df.to_csv(results_dir(cfg, "pipeline") / "decision_breakdown_test.csv", index=False); br_df

## 5. Precision–recall frontier (test)

In [ ]:
rows_t = {DET_ONLY: sweep(DET_ONLY, MT[DET_ONLY], T, cfg), PROPOSED: sweep(PROPOSED, MT[PROPOSED], T, cfg)}
plot_pr(rows_t, highlight={"DVD best-F1": res["DVD best-F1"], "Detector tau*": res["Detector only (tau*)"]},
        title="Test PR frontier (selection made on held-out)")
plt.savefig(results_dir(cfg, "figures") / "pr_frontier_pipeline_test.png", dpi=150, bbox_inches="tight"); plt.show()
pd.DataFrame(rows_t[PROPOSED]).to_csv(results_dir(cfg, "pipeline") / "dvd_sweep_test.csv", index=False)

## 6. (Optional) LLM aggregator instead of the rule
The verifier's own language model (text-only) receives c, the verdict and P(yes) and answers accept/reject. Compared with the rule on the same held-out subset.

In [ ]:
RUN_LLM_AGGREGATOR = False
if RUN_LLM_AGGREGATOR:
    from dvd.agents.verifier import build_verifier
    from dvd.agents.decision import LLMAggregatorAgent, rule_accept
    from dvd.metrics import DetectionEvaluator
    from tqdm.auto import tqdm
    agg = LLMAggregatorAgent(build_verifier(cfg))
    sub = HELDOUT[: cfg["decision"]["llm_aggregator_images"]]
    ev = DetectionEvaluator({i: H.det[i] for i in sub}, {i: H.gts[i] for i in sub}, cfg["coco"]["iou_threshold"])
    v, p = MH[PROPOSED]["verdicts"], MH[PROPOSED]["p_yes"]
    tl, th = dvd_best["tau_low"], dvd_best["tau_high"]
    m_llm = {}
    for i in tqdm(sub):
        c = ev.scores(i); m = c >= th; band = np.where((c >= tl) & (c < th))[0]
        if len(band):
            m[band] = agg.decide([H.det[i]["labels"][k] for k in band], c[band], v[i][band], p[i][band])
        m_llm[i] = m
    m_rule = {i: rule_accept(ev.scores(i), v[i], tl, th) for i in sub}
    display(pd.DataFrame([{"decision": "rule", **ev.evaluate(m_rule)}, {"decision": "LLM aggregator", **ev.evaluate(m_llm)}]))

## 7. Online pipeline demo (gated: only the uncertain band is verified)

In [ ]:
from dvd.agents.detector import build_detector
from dvd.agents.verifier import build_verifier
from dvd.agents.decision import RuleDecisionAgent
from dvd.pipeline import DetectVerifyDecide
from dvd.viz import show_detections
verifier = agg.verifier if RUN_LLM_AGGREGATOR else build_verifier(cfg)   # never load two VLMs on a T4
pipe = DetectVerifyDecide(build_detector(cfg, coco.cat_names, coco.name_to_cat_id), verifier,
                          RuleDecisionAgent(dvd_best["tau_low"], dvd_best["tau_high"]), cfg)
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, iid in zip(axes.ravel(), TEST[10:14]):
    img = coco.image(iid); r = pipe(img)
    keep = [k for k, s in enumerate(r["scores"]) if s >= dvd_best["tau_low"]]
    st = ["accepted" if r["accept"][k] else "rejected" for k in keep]
    show_detections(img, [r["boxes"][k] for k in keep], [r["labels"][k] for k in keep], [r["scores"][k] for k in keep], st, ax=ax,
                    title=f"{iid}: {r['verify_invocations']} crops verified, {r['total_time_s']:.2f}s")
plt.tight_layout(); plt.show()